# Clase 1 — Práctica: El Perceptrón desde Cero

**Objetivo:** entender la pieza más básica de toda IA construyéndola a mano, sin librerías.

Vas a:
1. Crear una **neurona artificial** en Python puro.
2. Entrenarla para que aprenda la función **AND**.
3. Entrenarla para que aprenda la función **OR**.
4. Probar con **XOR** y ver que **no puede** aprenderla.
5. Mostrar la idea de **tokens** (cómo el texto se vuelve números).

> **Solo usamos `random` y `math`.** No hace falta instalar nada.
> Si tenés matplotlib, se ven gráficos; si no, se muestra una versión ASCII.

## 1. Importar lo mínimo

Solo dos cosas de la biblioteca estándar de Python:
- `random` — para inicializar los pesos al azar.
- `math` — por si queremos funciones extra (no es estrictamente necesario aquí).

Intentamos importar matplotlib, pero si no está, no pasa nada: usamos fallback ASCII.

In [ ]:
import random
import math

try:
    import matplotlib.pyplot as plt
    TIENE_MPL = True
except ImportError:
    TIENE_MPL = False
    print("matplotlib no esta instalado: se usara fallback ASCII (todo igual de claro).")

## 2. La neurona artificial

Repasando la idea de los apuntes:

```
   input x1 --[peso w1]--+
                         +--> SUMA --> ¿pasa umbral? --> 1 o 0
   input x2 --[peso w2]--+
                         + sesgo (bias)
```

La neurona hace tres cosas:
1. **Suma ponderada:** multiplica cada input por su peso y suma todo + el sesgo.
2. **Función de activación:** si la suma >= 0, da `1`; si no, da `0` (función escalón).
3. **Predecir:** junta las dos anteriores.

Y para **aprender** (el perceptrón de Rosenblatt):
- Predice.
- Compara con la respuesta correcta (error).
- Ajusta los pesos un poquito en la dirección correcta.
- Repite muchas veces (epochs).

In [ ]:
class Neurona:
    """Una neurona artificial (perceptron) que aprende con la regla del perceptron."""

    def __init__(self, n_inputs, pesos=None, sesgo=None):
        # Si no nos dan pesos, los inicializamos al azar (entre -1 y 1)
        if pesos is None:
            self.pesos = [random.uniform(-1, 1) for _ in range(n_inputs)]
        else:
            self.pesos = pesos[:]
        # Lo mismo con el sesgo (bias)
        if sesgo is None:
            self.sesgo = random.uniform(-1, 1)
        else:
            self.sesgo = sesgo

    def suma_ponderada(self, inputs):
        """Multiplica cada input por su peso y suma todo + el sesgo."""
        return sum(w * x for w, x in zip(self.pesos, inputs)) + self.sesgo

    def activacion(self, z):
        """Funcion escalon: si z >= 0 da 1, si no da 0."""
        return 1 if z >= 0 else 0

    def predecir(self, inputs):
        """Junta suma ponderada + activacion."""
        return self.activacion(self.suma_ponderada(inputs))

    def entrenar(self, datos, epochs=10, lr=0.1, verbose=False):
        """
        Aprende ajustando pesos con cada error.

        datos: lista de (inputs, respuesta_esperada)
        epochs: cuantas veces recorremos todos los ejemplos
        lr: learning rate (que tanto ajustamos en cada paso)
        verbose: si mostramos el progreso
        """
        errores_historial = []
        for epoch in range(epochs):
            errores = 0
            for inputs, esperado in datos:
                pred = self.predecir(inputs)
                error = esperado - pred
                if error != 0:
                    # Ajustamos cada peso: peso_nuevo = peso_viejo + lr * error * input
                    for i in range(len(self.pesos)):
                        self.pesos[i] += lr * error * inputs[i]
                    self.sesgo += lr * error
                    errores += 1
            errores_historial.append(errores)
            if verbose:
                print(f"Epoch {epoch+1}: errores={errores} "
                      f"pesos={[round(p,3) for p in self.pesos]} "
                      f"sesgo={round(self.sesgo,3)}")
        return errores_historial

## 3. Entrenar con AND

**AND** ("y" lógico): la neurona debe dar `1` **solo si los dos inputs son 1**.

| x1 | x2 | salida |
|----|----|--------|
| 0  | 0  | 0      |
| 0  | 1  | 0      |
| 1  | 0  | 0      |
| 1  | 1  | 1      |

Vas a ver cómo los errores bajan epoch a epoch hasta llegar a 0.

In [ ]:
random.seed(42)  # para que los resultados sean reproducibles
n_and = Neurona(2)

datos_and = [
    ([0, 0], 0),
    ([0, 1], 0),
    ([1, 0], 0),
    ([1, 1], 1),
]

print("Pesos iniciales:", [round(p, 3) for p in n_and.pesos],
      "Sesgo inicial:", round(n_and.sesgo, 3))
print()

hist_and = n_and.entrenar(datos_and, epochs=15, lr=0.1, verbose=True)

print()
print("Pesos finales:", [round(p, 3) for p in n_and.pesos],
      "Sesgo final:", round(n_and.sesgo, 3))
print()
print("Resultados:")
for inputs, esperado in datos_and:
    print(f"  {inputs} -> {n_and.predecir(inputs)} (esperado {esperado})")

### ¿Aprendió AND?

Si los errores llegaron a 0 y las 4 predicciones coinciden con lo esperado, **¡la neurona aprendió AND!**

Mirá cómo los pesos fueron cambiando poco a poco. Al principio se equivocaba; al final, acierta. **Eso es aprender.** No magia: ajustar números repetidamente.

## 4. Entrenar con OR

**OR** ("o" lógico): la neurona da `1` si **al menos uno** de los inputs es 1.

| x1 | x2 | salida |
|----|----|--------|
| 0  | 0  | 0      |
| 0  | 1  | 1      |
| 1  | 0  | 1      |
| 1  | 1  | 1      |

In [ ]:
random.seed(7)
n_or = Neurona(2)

datos_or = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 1),
]

print("Pesos iniciales:", [round(p, 3) for p in n_or.pesos],
      "Sesgo inicial:", round(n_or.sesgo, 3))
print()

hist_or = n_or.entrenar(datos_or, epochs=15, lr=0.1, verbose=True)

print()
print("Pesos finales:", [round(p, 3) for p in n_or.pesos],
      "Sesgo final:", round(n_or.sesgo, 3))
print()
print("Resultados:")
for inputs, esperado in datos_or:
    print(f"  {inputs} -> {n_or.predecir(inputs)} (esperado {esperado})")

## 5. El límite: XOR

**XOR** ("o exclusivo"): la neurona da `1` si **exactamente uno** de los inputs es 1 (pero no los dos).

| x1 | x2 | salida |
|----|----|--------|
| 0  | 0  | 0      |
| 0  | 1  | 1      |
| 1  | 0  | 1      |
| 1  | 1  | 0      |

Entrená y mirá qué pasa. **Spoiler:** no va a poder.

In [ ]:
random.seed(99)
n_xor = Neurona(2)

datos_xor = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 0),
]

print("Entrenando XOR por 30 epochs...\n")
hist_xor = n_xor.entrenar(datos_xor, epochs=30, lr=0.1, verbose=False)

print("Errores por epoch:")
for i, e in enumerate(hist_xor):
    print(f"  Epoch {i+1}: {e} errores")

print()
aciertos = sum(1 for inputs, esperado in datos_xor
               if n_xor.predecir(inputs) == esperado)
print(f"Aciertos finales: {aciertos}/4")
print()
print("Resultados:")
for inputs, esperado in datos_xor:
    print(f"  {inputs} -> {n_xor.predecir(inputs)} (esperado {esperado})")

### ¿Por qué XOR no se puede?

El perceptrón **traza una línea recta** para separar los `1` de los `0`.

- **AND:** los `1` están agrupados en una esquina. Una línea los separa. ✅
- **OR:** los `1` están en tres esquinas. Una línea los separa. ✅
- **XOR:** los `1` están en **esquinas opuestas** (diagonal). **No hay una línea recta** que separe `1` de `0`. ❌

```
   AND (separable)        XOR (NO separable)
   1  0  0  0              1  0  1  0
      \___ linea               no hay linea
   0  0  0  1              0  1  0  1
```

Para resolver XOR hacen falta **varias neuronas** juntas (una red multicapa). Ese es el siguiente paso en la historia de la IA. Por ahora, lo importante es que entiendas el **límite** del perceptrón.

## 6. Ver los errores bajar (AND vs OR vs XOR)

Comparemos los tres entrenamientos. Si tenés matplotlib, se ve un gráfico; si no, una tablita ASCII.

In [ ]:
if TIENE_MPL:
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(hist_and)+1), hist_and, 'bo-', label='AND')
    plt.plot(range(1, len(hist_or)+1), hist_or, 'gs-', label='OR')
    plt.plot(range(1, len(hist_xor)+1), hist_xor, 'r^-', label='XOR')
    plt.xlabel('Epoch')
    plt.ylabel('Errores')
    plt.title('Errores por epoch: AND y OR llegan a 0, XOR no')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Epoch | AND | OR | XOR")
    print("------+-----+----+----")
    for i in range(max(len(hist_and), len(hist_or), len(hist_xor))):
        a = hist_and[i] if i < len(hist_and) else "-"
        o = hist_or[i] if i < len(hist_or) else "-"
        x = hist_xor[i] if i < len(hist_xor) else "-"
        print(f"  {i+1:2d}  |  {a}  |  {o} |  {x}")
    print()
    print("AND y OR llegan a 0 errores. XOR se queda estancado. Ese es el limite del perceptron.")

## 7. Del perceptrón a los tokens

La neurona solo entiende **números**. Pero la IA que usamos hoy (ChatGPT, etc.) trabaja con **texto**. ¿Cómo metemos texto en una neurona?

→ **Tokens.** Un token es un trozo mínimo de texto con un número (ID) asociado.

Acá vas a ver un mini-tokenizador: se parte un texto en palabras y se le asigna un ID a cada una. Eso es lo que hacen los modelos reales (aunque con trozos más finos, no solo palabras enteras).

In [ ]:
texto = "Hola mundo IA"

# "Tokenizamos": partimos en palabras y les damos un ID unico
vocab = {}        # palabra -> ID
id_actual = 1000  # los IDs empiezan en 1000 (como quieran)
tokens = []       # lista de IDs

for palabra in texto.split():
    if palabra not in vocab:
        vocab[palabra] = id_actual
        id_actual += 1
    tokens.append(vocab[palabra])

print(f"Texto original:  {texto!r}")
print(f"Palabras:        {texto.split()}")
print(f"Tokens (IDs):    {tokens}")
print(f"Vocabulario:     {vocab}")
print()
print("Ahora cada palabra es un numero. Esa lista de numeros SI la puede procesar una neurona.")

### Lo que acabás de hacer

1. Partimos el texto en piezas (acá, palabras; los modelos reales usan piezas más finas).
2. Le asignamos un número único a cada pieza.
3. El texto se convirtió en una **lista de números**.

Eso es lo que hace un **tokenizador**. Y esa lista de números es lo que entra a las neuronas.

> **No se profundiza más en esta clase.** En la **Clase 4** se ve cómo se miden los tokens, cuánto cuesta cada uno, qué es la longitud de contexto y la latencia. Por ahora, con la idea basta: **texto → tokens → números → neuronas.**

## 8. Resumen

| Qué | Resultado |
|-----|-----------|
| AND | ✅ La neurona aprende (errores → 0) |
| OR  | ✅ La neurona aprende (errores → 0) |
| XOR | ❌ La neurona NO puede (necesita más neuronas) |
| Tokens | Texto → números que la neurona entiende |

**Ideas para llevarte:**
- Una neurona **aprende** ajustando pesos. No magia: repetición.
- El perceptrón tiene un **límite** (XOR). Para superarlo, redes multicapa.
- La IA trabaja con **números**. Los tokens son el puente entre texto y números.

## 9. Ejercicios (para hacer en casa)

1. **Cambiá la semilla** (`random.seed(...)`) y volvé a entrenar AND. ¿Llega a 0 errores igual? ¿En cuántos epochs?

2. **Cambiá el learning rate** (`lr=0.5` en lugar de `0.1`). ¿Aprende más rápido o se vuelve inestable?

3. **Inventá tu propia función** booleana (ej: "da 1 solo si x1=1 y x2=0"). ¿La puede aprender el perceptrón?

4. **Probá con 3 inputs** (ej: AND de 3 variables). Cambiá `Neurona(2)` por `Neurona(3)` y agregá los datos.

5. **Tokenizá una frase más larga.** Modificá la variable `texto` y mirá los IDs que se generan.

> Si te trabás en alguno, no te frustres: eso es programar. Podés consultarlo en la clase de consulta.